# vi-function-calling-slm · 02 SFT (QLoRA) Qwen3-1.7B

Cần **GPU T4**. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + output của notebook 01 (`data/gen/*.jsonl`). Kaggle: add dataset `vi-function-calling-slm-code`
và output notebook 01 làm input; Colab: chạy notebook 01 trước, dữ liệu nằm sẵn trên Drive.
Secret: `HF_TOKEN` (đẩy adapter lên `<tài khoản HF>/vi-fc-qwen3-1.7b`), thiếu thì bỏ qua bước push.
Trên Colab trainer lưu checkpoint vào `outputs/sft-<hash dữ liệu>` trên Drive, bị ngắt thì chạy lại notebook là train tiếp.

- Model: Qwen3-1.7B (Apache-2.0), load 4-bit, LoRA r=16 trên mọi lớp linear. Có Unsloth thì dùng (nhanh ~2x),
  cài/import lỗi thì tự chuyển sang transformers + peft + bitsandbytes
- Định dạng: chat template gốc của Qwen3 với `tools=`, `enable_thinking=False`
- Chỉ tính loss trên lượt trả lời cuối của assistant (system + tools + lượt user bị mask)
- Tối đa 2 epoch, số step chốt theo ngân sách ~40 phút trên T4, quá 45 phút thì dừng cứng

In [ ]:
import glob, json, os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # Kaggle T4 x2: train trên 1 GPU, không để Trainer bọc DataParallel
IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-function-calling-slm" if IN_COLAB else os.path.abspath("vi-function-calling-slm")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-function-calling-slm {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    WORK = os.getcwd()
    if IN_COLAB:
        from google.colab import drive
        try:  # không ai bấm cho phép Drive thì sau 2 phút chạy trên /content, output trước lấy từ HF Hub
            drive.mount("/content/drive", timeout_ms=120_000)
            WORK = "/content/drive/MyDrive/ai-portfolio/vi-function-calling-slm"
        except Exception as e:
            print("không mount được Drive, chạy trên /content/work:", e)
            WORK = "/content/work"
NO_DRIVE = WORK == "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


if NO_DRIVE and not os.path.exists(f"{WORK}/data/gen/train.jsonl"):
    from vi_fc.hub import pull
    os.environ.setdefault("HF_TOKEN", secret("HF_TOKEN") or "")
    pull("vi-fc-synth", f"{WORK}/data/gen", repo_type="dataset")  # notebook 01 đã đẩy lên (repo private)

DATA = os.path.dirname(find("data/gen/train.jsonl", "chạy notebook 01 trước"))
print(CODE, DATA)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
# khoá bản torch có sẵn: pip mà tự nâng torch thì torchvision/torchaudio lệch bản, import transformers hỏng theo
!pip list --format=freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/keep-torch.txt
!timeout 900 pip install -q -c /tmp/keep-torch.txt unsloth || echo "cài unsloth lỗi, dùng transformers + peft"
!pip install -q -c /tmp/keep-torch.txt bitsandbytes peft trl datasets accelerate

token = secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
else:
    print("thiếu HF_TOKEN, sẽ không push")
if secret("HF_USER"):  # tuỳ chọn: đẩy vào tài khoản/org khác với chủ HF_TOKEN
    os.environ["HF_USER"] = secret("HF_USER")

from vi_fc.train import has_unsloth

UNSLOTH = has_unsloth()
print("unsloth:", UNSLOTH)

In [ ]:
if UNSLOTH:
    import unsloth  # noqa: F401  import trước transformers để nó patch kịp
import torch
from vi_fc.train import BASE, UNSLOTH_BASE, add_lora, load_4bit

MAX_LEN = 2048  # ví dụ dài nhất ~1.8k token, phần lớn là system + schema 17 tool (~1.6k)
model, tokenizer = load_4bit(UNSLOTH_BASE if UNSLOTH else BASE, MAX_LEN, UNSLOTH)
model = add_lora(model, UNSLOTH)
tokenizer.padding_side = "right"

## Dữ liệu -> token theo chat template, mask phần prompt
Xem độ dài: phần system + schema 17 tool chiếm phần lớn, ảnh hưởng thẳng tới độ trễ prefill trên CPU.

In [ ]:
from datasets import Dataset
from vi_fc.prompt import build_messages, record_messages, split_prompt_completion
from vi_fc.tools import openai_tools
from vi_fc.train import tokenize_sft

TOOLS = openai_tools()


def load(name):
    return [json.loads(l) for l in open(f"{DATA}/{name}.jsonl", encoding="utf-8") if l.strip()]


def apply(msgs, gen):
    return tokenizer.apply_chat_template(msgs, tools=TOOLS, tokenize=False, add_generation_prompt=gen,
                                         enable_thinking=False)


def encode(recs):
    rows = [tokenize_sft(tokenizer, *split_prompt_completion(apply, record_messages(r)), MAX_LEN) for r in recs]
    if None in rows:
        print(f"bỏ {rows.count(None)} ví dụ dài hơn {MAX_LEN} token")
    return Dataset.from_list([r for r in rows if r])


train, val = load("train"), load("val")
ds_train = encode(train).shuffle(seed=0)
ds_val = encode(val[:100])  # eval loss lúc train, 100 câu đủ thấy xu hướng

prefix = apply(build_messages("x", None), True)
lens = sorted(len(x) for x in ds_train["input_ids"])
print(f"{len(train)} train / {len(val)} val")
print("token prompt cố định (system + tools):", len(tokenizer(prefix).input_ids))
print(f"token/ví dụ p50={lens[len(lens)//2]} max={lens[-1]}")
# kiểm tra mask: phần còn lại sau khi bỏ -100 phải chỉ là câu trả lời của assistant
ex = ds_train[0]
print(repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

In [ ]:
import hashlib

from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from vi_fc.train import TPS, fp_flags, plan_steps, time_budget

BS, GA = 2, 8
# step chốt theo số ví dụ + ngân sách 40 phút (ước lượng tốc độ T4); chậm hơn dự kiến thì time_budget cắt ở 45 phút
max_steps = plan_steps(len(ds_train), BS * GA, 2, sum(lens) / len(lens), TPS[UNSLOTH], minutes=40)
print(f"{max_steps} step ≈ {max_steps * BS * GA / len(ds_train):.2f} epoch")
# checkpoint theo dữ liệu: sinh thêm dữ liệu rồi chạy lại thì train mới, không resume nhầm checkpoint cũ
RUN = "outputs/sft-" + hashlib.md5(open(f"{DATA}/train.jsonl", "rb").read()).hexdigest()[:8]
args = TrainingArguments(
    output_dir=RUN, per_device_train_batch_size=BS, gradient_accumulation_steps=GA, max_steps=max_steps,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=max(1, max_steps // 30), weight_decay=0.01,
    optim="adamw_8bit", **fp_flags(), logging_steps=10,
    eval_strategy="steps" if len(ds_val) else "no", eval_steps=50, per_device_eval_batch_size=2,
    prediction_loss_only=True,  # không gom logits (vocab 152k) lúc eval, không thì tràn RAM
    # Colab có thể ngắt giữa chừng -> lưu checkpoint lên Drive để train tiếp; Kaggle chạy một mạch nên không lưu
    save_strategy="steps" if IN_COLAB else "no", save_steps=50, save_total_limit=1,
    report_to="none", seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val if len(ds_val) else None,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8),
                  callbacks=[time_budget(45)])

In [ ]:
stats = trainer.train(resume_from_checkpoint=bool(glob.glob(f"{RUN}/checkpoint-*")))
print(stats.metrics)
# lưu adapter ngay, trước phần eval: eval lỗi cũng không mất kết quả train
model.save_pretrained("vi-fc-sft-lora")
tokenizer.save_pretrained("vi-fc-sft-lora")
from vi_fc.hub import push

# đẩy adapter lên Hub ngay, trước phần eval: chạy không có Drive thì notebook 03 lấy từ đây
push("vi-fc-sft-lora", "vi-fc-qwen3-1.7b", path_in_repo="sft-lora")

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
for pts, style, label in ((tr, "-", "train"), (ev, "o-", "val")):
    if pts:
        plt.plot(*zip(*pts), style, label=label)
plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.title("SFT loss")
plt.savefig("sft_loss.png", dpi=110, bbox_inches="tight")

## Đánh giá nhanh trên val (50 câu) và bộ viết tay
Đánh giá đầy đủ trên tập test ở notebook 04.

In [ ]:
from vi_fc.evaluate import compute_metrics, load_jsonl, predict
from vi_fc.inference import FunctionCaller
from vi_fc.train import set_mode

set_mode(model, UNSLOTH, train=False)
fc = FunctionCaller("hf", model, tokenizer=tokenizer, guard=False, max_tokens=128)
manual = load_jsonl(find("data/test_manual.jsonl", "thiếu data/test_manual.jsonl của repo"))
for name, recs in (("val", val[:50]), ("manual", manual)):
    rows = predict(recs, fc, "sft", name, f"sft_{name}.jsonl")
    m = compute_metrics(rows)
    print(name, json.dumps({k: v for k, v in m.items() if k != "exact_by_scenario"}, ensure_ascii=False))
    print("  theo kịch bản:", m["exact_by_scenario"])

In [ ]:
# soi các câu sai trên bộ viết tay
from vi_fc.match import actions_match
from vi_fc.parse import Action

for r in load_jsonl("sft_manual.jsonl"):
    if not actions_match(Action.from_dict(r["gold"]), Action.from_dict(r["pred"])):
        print(r["user"]); print("   gold:", r["gold"]); print("   pred:", r["raw"][:200])